# 0. Limpieza de datos

In [24]:
from pathlib import Path  

import pandas as pd

root = Path.cwd()
excel = root / "data" / "raw" / "2026 BASE RESERVAS.xlsx"

emision = pd.read_excel(excel, sheet_name="EMISION")
sinestralidad = pd.read_excel(excel, sheet_name="SINIESTRALIDAD")

In [25]:
emision.info()

<class 'pandas.DataFrame'>
RangeIndex: 21353 entries, 0 to 21352
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   año                21353 non-null  int64
 1   tipo_embarcacion   21353 non-null  str  
 2   uso                21353 non-null  str  
 3   anio_construccion  21353 non-null  int64
 4   prima_emitida      21353 non-null  int64
 5   prima_retenida     21353 non-null  int64
 6   comision_directa   21353 non-null  int64
dtypes: int64(5), str(2)
memory usage: 1.1 MB


In [26]:
sinestralidad.info()

<class 'pandas.DataFrame'>
RangeIndex: 2763 entries, 0 to 2762
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   año                2763 non-null   int64  
 1   tipo_embarcacion   2763 non-null   str    
 2   uso                2763 non-null   str    
 3   anio_construccion  2739 non-null   float64
 4   cobertura          2763 non-null   str    
 5   causa_siniestro    2763 non-null   str    
 6   0_monto pagado     2763 non-null   float64
 7   1_monto pagado     2763 non-null   float64
 8   2_monto pagado     2763 non-null   float64
 9   3_monto pagado     2763 non-null   float64
dtypes: float64(5), int64(1), str(4)
memory usage: 216.0 KB


In [27]:
from funciones_reservas import normalizar_categorias

for nombre, df in [("emision", emision), ("sinestralidad", sinestralidad)]:
    columnas = [col for col in ["tipo_embarcacion", "uso"] if col in df]
    if columnas:
        df_actual = normalizar_categorias(df, columnas, minusculas=True)    # Normalizar categorías de las columnas especificadas
        df.loc[:, columnas] = df_actual[columnas]
    if "anio_construccion" in df:
        df["anio_construccion"] = pd.to_numeric(df["anio_construccion"], errors="coerce").astype("Int64")

In [28]:
from IPython.display import display, HTML

from funciones_reservas import resumen_calidad

tablas = []

for nombre, df in [("Emisión", emision), ("sinestralidad", sinestralidad)]:
    resumen = resumen_calidad(df)

    tablas.append(f"""
    <div style="flex: 1; min-width: 0;">
        <h3>{nombre}</h3>
        {resumen[["unicos", "ceros", "negativos"]].to_html()}
    </div>
    """)

display(HTML(
    '<div style="display: flex; gap: 30px; align-items: flex-start;">'
    + "".join(tablas) +
    "</div>"
))

,unicos,ceros,negativos
año,11,0,0
tipo_embarcacion,32,—,—
uso,11,—,—
anio_construccion,90,0,0
prima_emitida,18755,0,1126
prima_retenida,15882,2009,938
comision_directa,11827,2226,857
,unicos,ceros,negativos
año,11,0,0
tipo_embarcacion,26,—,—


# 1. Factores Utilizados

In [29]:
from os import getenv

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

token = getenv("BANXICO_API")
url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SP1/datos/2015-01-01/2025-12-31"

resp = requests.get(url, headers={"Bmx-Token": token}, timeout=30)
resp.raise_for_status()

datos = resp.json()["bmx"]["series"][0]["datos"]
df = pd.DataFrame(datos)

serie_inpc = pd.Series(
    data=pd.to_numeric(df["dato"], errors="coerce").to_numpy(),
    index=pd.to_datetime(df["fecha"], format="%d/%m/%Y").rename("fecha"),
    name="inpc",
)

In [30]:
from funciones_reservas import ajustar_inpc, normalizar_categorias

# Actualizar primas y comisiones según su año de emisión
emision_ajustada = ajustar_inpc(
    emision,
    ["prima_emitida", "prima_retenida", "comision_directa"],
    "año", 2025, serie_inpc
)


# Actualizar los montos de siniestros según su año de pago
sinestralidad_ajustada = sinestralidad.copy()

for desarrollo in range(4):

    columna = f"{desarrollo}_monto pagado"

    # Calcular el año correspondiente a cada desarrollo
    sinestralidad_ajustada["año_pago"] = sinestralidad_ajustada["año"] + desarrollo

    # Considerar únicamente los pagos hasta 2025
    mascara = sinestralidad_ajustada["año_pago"] <= 2025

    # Ajustar los montos del desarrollo correspondiente
    datos = ajustar_inpc(
        sinestralidad_ajustada.loc[mascara, ["año_pago", columna]],
        columna, "año_pago", 2025, serie_inpc
    )

    sinestralidad_ajustada[f"{columna}_ajustada"] = datos[f"{columna}_ajustada"]

# Eliminar la columna temporal
sinestralidad_ajustada.drop(columns="año_pago", inplace=True)

# Mostrar una vista previa de los resultados
display(emision_ajustada.head(), sinestralidad_ajustada.head())

/var/folders/_7/s1cs5hwj3dg862vk2w55zkxr0000gn/T/ipykernel_48208/3504354149.py:33: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  sinestralidad_ajustada.drop(columns="año_pago", inplace=True)


,año,tipo_embarcacion,uso,anio_construccion,prima_emitida,prima_retenida,comision_directa,factor_inpc,prima_emitida_ajustada,prima_retenida_ajustada,comision_directa_ajustada
0,2015,lancha,comercial,2007,3219,3219,644,1.606368,5170.900076,5170.900076,1034.501289
1,2015,buque-tanque,comercial,1992,332844,332844,66569,1.606368,534670.104079,534670.104079,106934.342090
2,2015,velero,privado y/o placer,2012,25759,25759,5151,1.606368,41378.445190,41378.445190,8274.403943
3,2015,abastecedor,privado y/o placer,2001,-37,-37,-7,1.606368,-59.435633,-59.435633,-11.244579
4,2015,lancha,pesca,2013,-194,-194,-31,1.606368,-311.635481,-311.635481,-49.797422


,año,tipo_embarcacion,uso,anio_construccion,cobertura,causa_siniestro,0_monto pagado,1_monto pagado,2_monto pagado,3_monto pagado,0_monto pagado_ajustada,1_monto pagado_ajustada,2_monto pagado_ajustada,3_monto pagado_ajustada
0,2015,abastecedor,comercial,1979,Todo Riesgo (Casco),Incendio,29665062.0,0.0,0.0,0.0,4.765302e+07,0.0,0.0,0.0
1,2015,abastecedor,comercial,2011,Todo Riesgo (Casco),Averia particular,15502388.0,0.0,0.0,0.0,2.490255e+07,0.0,0.0,0.0
2,2015,buque-tanque,comercial,2013,Todo Riesgo (Casco),Varadura,10377414.0,0.0,0.0,0.0,1.666995e+07,0.0,0.0,0.0
3,2015,buque-tanque,comercial,2014,Todo Riesgo (Casco),Negligencia de la tripulacion,9721320.0,0.0,0.0,0.0,1.561602e+07,0.0,0.0,0.0
4,2015,remolcador,remolcador,2013,Todo Riesgo (Casco),Averia particular,9668053.0,0.0,0.0,0.0,1.553046e+07,0.0,0.0,0.0


# 2. Análisis de variables

## 2.1. Prima Emitida

### 2.1.1. Mercado total: monto anual y crecimiento de la prima emitida

Se analizan el monto anual de prima emitida, su variación monetaria y su crecimiento real interanual. También se calcula el CAGR y la pendiente de la tendencia lineal en pesos constantes de diciembre de 2025.


In [31]:
# Paleta y estilo gráfico institucional
import plotly.io as pio

COLORES = {
    "navy": "#2A4163",
    "azul": "#5B8FF9",
    "naranja": "#F58518",
    "azul_claro": "#C7D6E6",
    "verde": "#54A24B",
    "violeta": "#B279A2",
    "gris": "#9C9C9C",
    "gris_claro": "#E5E7EB",
    "fondo": "#F2F2F2",
}

COLORES_TIPOS = [
    "#5B8FF9", "#F58518", "#54A24B", "#B279A2",
    "#E45756", "#72B7B2", "#4C78A8", "#FF9DA6",
    "#9C9C9C", "#2A4163", "#BAB0AC", "#B5BD61",
]

pio.templates["reservas"] = pio.templates["plotly_white"]
pio.templates["reservas"].layout.update(
    paper_bgcolor=COLORES["fondo"],
    plot_bgcolor=COLORES["fondo"],
    font=dict(family="Arial", size=13, color=COLORES["navy"]),
    title=dict(font=dict(size=20, color=COLORES["navy"]), x=0.03, xanchor="left"),
    legend=dict(font=dict(size=12), orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(l=65, r=40, t=110, b=65),
    colorway=COLORES_TIPOS,
)
pio.templates["reservas"].layout.xaxis.update(
    showgrid=False,
    zeroline=False,
    showline=False,
)
pio.templates["reservas"].layout.yaxis.update(
    gridcolor="#FFFFFF",
    gridwidth=1,
    zeroline=False,
    showline=False,
)
pio.templates.default = "reservas"

import numpy as np
import plotly.graph_objects as go

# Tabla anual del mercado
columnas = [
    "prima_emitida_ajustada",
    "comision_directa_ajustada",
    "prima_retenida_ajustada",
]
mercado = (
    emision_ajustada.groupby("año", as_index=True)[columnas]
    .sum()
    .sort_index()
)

mercado["variacion_monetaria"] = mercado["prima_emitida_ajustada"].diff()
mercado["crecimiento_real"] = mercado["prima_emitida_ajustada"].pct_change() * 100

primer_anio = mercado.index.min()
ultimo_anio = mercado.index.max()
periodos = ultimo_anio - primer_anio
cagr = (
    mercado.loc[ultimo_anio, "prima_emitida_ajustada"]
    / mercado.loc[primer_anio, "prima_emitida_ajustada"]
) ** (1 / periodos) - 1
cagr *= 100

coeficiente = np.polyfit(
    mercado.index.astype(float),
    mercado["prima_emitida_ajustada"],
    1,
)
pendiente = coeficiente[0]
tendencia = np.polyval(coeficiente, mercado.index.astype(float))

tabla_emitida = mercado[[
    "prima_emitida_ajustada",
    "variacion_monetaria",
    "crecimiento_real",
]].rename(columns={
    "prima_emitida_ajustada": "Prima emitida",
    "variacion_monetaria": "Variación monetaria",
    "crecimiento_real": "Crecimiento real (%)",
})

display(tabla_emitida.style.format({
    "Prima emitida": "{:,.2f}",
    "Variación monetaria": "{:,.2f}",
    "Crecimiento real (%)": "{:.2f}",
}))
print(f"TCAC/CAGR: {cagr:.2f}%")
print(f"Pendiente lineal: {pendiente:,.2f} pesos/año")

anio_max = mercado["crecimiento_real"].idxmax()
anio_min = mercado["crecimiento_real"].idxmin()
print(
    f"Máximo crecimiento: {mercado.loc[anio_max, 'crecimiento_real']:.2f}% en {anio_max}; "
    f"mínimo: {mercado.loc[anio_min, 'crecimiento_real']:.2f}% en {anio_min}."
)

fig = go.Figure()
fig.add_bar(
    x=mercado.index,
    y=mercado["prima_emitida_ajustada"],
    name="Prima emitida",
    marker_color=COLORES["azul"],
)
fig.add_scatter(
    x=mercado.index,
    y=tendencia,
    mode="lines",
    name="Tendencia lineal",
    line=dict(color=COLORES["navy"], dash="dot", width=3),
)
fig.add_scatter(
    x=mercado.index,
    y=mercado["crecimiento_real"],
    mode="lines+markers",
    name="Crecimiento real",
    yaxis="y2",
    line=dict(color=COLORES["naranja"], width=2),
)
fig.update_layout(
    title=f"Prima Emitida y crecimiento real — CAGR {cagr:.2f}%",
    template="reservas",
    height=520,
    legend=dict(orientation="h", y=1.02, x=0),
    yaxis=dict(title="Prima emitida (pesos constantes)", tickformat="~s"),
    yaxis2=dict(title="Crecimiento (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis=dict(title="Año", dtick=1),
)
fig.show()


,Prima emitida,Variación monetaria,Crecimiento real (%)
año,,,
2015,"2,479,720,780.88",nan,nan
2016,"2,672,242,961.19","192,522,180.31",7.76
2017,"3,040,127,080.77","367,884,119.58",13.77
2018,"2,549,310,473.56","-490,816,607.21",-16.14
2019,"3,636,121,176.14","1,086,810,702.59",42.63
2020,"1,677,942,699.91","-1,958,178,476.23",-53.85
2021,"1,958,679,233.46","280,736,533.54",16.73
2022,"1,964,567,043.28","5,887,809.82",0.30
2023,"4,603,093,102.01","2,638,526,058.74",134.31


TCAC/CAGR: 9.51%
Pendiente lineal: 223,003,425.94 pesos/año
Máximo crecimiento: 134.31% en 2023; mínimo: -53.85% en 2020.


### 2.1.2. Comisiones directas respecto a la prima emitida

Se compara el monto anual de comisiones directas con el índice de comisión y su variación interanual en puntos porcentuales.


In [32]:
mercado["indice_comision"] = (
    mercado["comision_directa_ajustada"]
    / mercado["prima_emitida_ajustada"]
    * 100
)
mercado["variacion_comision_pp"] = mercado["indice_comision"].diff()

promedio_comision = mercado["indice_comision"].mean()
tabla_comisiones = mercado[[
    "comision_directa_ajustada",
    "indice_comision",
    "variacion_comision_pp",
]].rename(columns={
    "comision_directa_ajustada": "Comisiones directas",
    "indice_comision": "Índice de comisión (%)",
    "variacion_comision_pp": "Variación (pp)",
})

display(tabla_comisiones.style.format({
    "Comisiones directas": "{:,.2f}",
    "Índice de comisión (%)": "{:.2f}",
    "Variación (pp)": "{:.2f}",
}))
print(f"Promedio histórico del índice de comisión: {promedio_comision:.2f}%")

fig = go.Figure()
fig.add_scatter(
    x=mercado.index,
    y=mercado["indice_comision"],
    mode="lines+markers+text",
    text=mercado["indice_comision"].map(lambda valor: f"{valor:.1f}%"),
    textposition="top center",
    name="Índice de comisión",
    line=dict(color=COLORES["naranja"], width=3),
)
fig.add_hline(
    y=promedio_comision,
    line_dash="dot",
    line_color=COLORES["gris"],
    annotation_text=f"Promedio {promedio_comision:.1f}%",
)
fig.update_layout(
    title="Comisiones directas como porcentaje de la Prima Emitida",
    template="reservas",
    height=480,
    xaxis=dict(title="Año", dtick=1),
    yaxis=dict(title="Índice de comisión", ticksuffix="%"),
)
fig.show()


,Comisiones directas,Índice de comisión (%),Variación (pp)
año,,,
2015,"78,474,806.83",3.16,nan
2016,"66,635,305.28",2.49,-0.67
2017,"79,258,664.70",2.61,0.11
2018,"52,843,465.11",2.07,-0.53
2019,"79,994,586.42",2.20,0.13
2020,"65,891,289.13",3.93,1.73
2021,"86,724,899.65",4.43,0.50
2022,"80,758,320.72",4.11,-0.32
2023,"73,890,188.44",1.61,-2.51


Promedio histórico del índice de comisión: 2.72%


### 2.1.3. Primas retenidas respecto a la prima emitida

Se descompone la prima emitida en la parte retenida y la parte cedida, y se analiza el índice anual de retención.


In [33]:
mercado["prima_cedida"] = (
    mercado["prima_emitida_ajustada"]
    - mercado["prima_retenida_ajustada"]
)
mercado["indice_retencion"] = (
    mercado["prima_retenida_ajustada"]
    / mercado["prima_emitida_ajustada"]
    * 100
)
mercado["variacion_retencion_pp"] = mercado["indice_retencion"].diff()

tabla_retencion = mercado[[
    "prima_retenida_ajustada",
    "prima_cedida",
    "indice_retencion",
    "variacion_retencion_pp",
]].rename(columns={
    "prima_retenida_ajustada": "Prima retenida",
    "prima_cedida": "Prima cedida",
    "indice_retencion": "Índice de retención (%)",
    "variacion_retencion_pp": "Variación (pp)",
})

display(tabla_retencion.style.format({
    "Prima retenida": "{:,.2f}",
    "Prima cedida": "{:,.2f}",
    "Índice de retención (%)": "{:.2f}",
    "Variación (pp)": "{:.2f}",
}))

fig = go.Figure()
fig.add_bar(
    x=mercado.index,
    y=mercado["prima_retenida_ajustada"],
    name="Prima retenida",
    marker_color=COLORES["azul"],
)
fig.add_bar(
    x=mercado.index,
    y=mercado["prima_cedida"],
    name="Prima cedida",
    marker_color=COLORES["azul_claro"],
)
fig.add_scatter(
    x=mercado.index,
    y=mercado["indice_retencion"],
    mode="lines+markers+text",
    text=mercado["indice_retencion"].map(lambda valor: f"{valor:.1f}%"),
    textposition="top center",
    name="Índice de retención",
    yaxis="y2",
    line=dict(color=COLORES["naranja"], width=3),
)
fig.update_layout(
    title="Primas retenida y cedida, con índice de retención",
    template="reservas",
    barmode="stack",
    height=520,
    yaxis=dict(title="Prima (pesos constantes)", tickformat="~s"),
    yaxis2=dict(title="Retención (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis=dict(title="Año", dtick=1),
)
fig.show()


,Prima retenida,Prima cedida,Índice de retención (%),Variación (pp)
año,,,,
2015,"270,800,276.35","2,208,920,504.53",10.92,nan
2016,"261,742,686.75","2,410,500,274.44",9.79,-1.13
2017,"274,925,239.15","2,765,201,841.62",9.04,-0.75
2018,"172,496,638.80","2,376,813,834.75",6.77,-2.28
2019,"219,875,800.30","3,416,245,375.84",6.05,-0.72
2020,"211,365,113.09","1,466,577,586.82",12.60,6.55
2021,"220,334,925.41","1,738,344,308.05",11.25,-1.35
2022,"233,055,318.39","1,731,511,724.89",11.86,0.61
2023,"231,459,288.79","4,371,633,813.22",5.03,-6.83


### 2.1.4. Comportamiento anual por tipo de embarcación

Se identifica el número de tipos activos, su prima anual, los estadísticos por tipo, las entradas y salidas entre ejercicios, y los principales cambios de participación.


In [34]:
from IPython.display import clear_output
clear_output(wait=True)

# Monto anual positivo de prima emitida por tipo
anual_tipo = (
    emision_ajustada[emision_ajustada["prima_emitida_ajustada"] > 0]
    .groupby(["año", "tipo_embarcacion"])["prima_emitida_ajustada"]
    .sum()
    .unstack(fill_value=0)
)

activos = {
    year: set(anual_tipo.loc[year][anual_tipo.loc[year] > 0].index)
    for year in anual_tipo.index
}
tipos_activos = pd.Series(
    {year: len(tipos) for year, tipos in activos.items()},
    name="Tipos activos",
)

display(tipos_activos.to_frame().T)

entradas = {}
salidas = {}
for anterior, actual in zip(anual_tipo.index[:-1], anual_tipo.index[1:]):
    entradas[actual] = sorted(activos[actual] - activos[anterior])
    salidas[actual] = sorted(activos[anterior] - activos[actual])

for year in anual_tipo.index[1:]:
    print(
        f"{year}: entradas={entradas[year] or 'ninguna'}; "
        f"salidas={salidas[year] or 'ninguna'}"
    )


,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Tipos activos,26,26,25,27,26,27,26,26,26,24,25


2016: entradas=['barco-hidrografico', 'buque-tanque quimico']; salidas=['barco-escuela', 'tiburoneros']
2017: entradas=['barco-escuela']; salidas=['barco-hidrografico', 'militar u oficial']
2018: entradas=['barco-hidrografico', 'buque-tanque-minerales', 'militar u oficial']; salidas=['buque-tanque quimico']
2019: entradas=['porta contenedor']; salidas=['artefactos navales', 'buque transportador de autos']
2020: entradas=['artefactos navales', 'buque transportador de autos', 'tiburoneros']; salidas=['barco-hidrografico', 'porta contenedor']
2021: entradas=['carguero tipo roll on - roll off']; salidas=['buque transportador de autos', 'buque-tanque-minerales']
2022: entradas=ninguna; salidas=ninguna
2023: entradas=ninguna; salidas=ninguna
2024: entradas=['porta contenedor']; salidas=['barco-escuela', 'carguero tipo roll on - roll off', 'escamero']
2025: entradas=['barco-escuela']; salidas=ninguna


#### Estadísticos y valores atípicos por tipo

In [35]:
from IPython.display import clear_output
clear_output(wait=True)

# Estadísticos y valores atípicos por tipo
serie_tipo = anual_tipo.stack().rename("Prima anual").reset_index()
serie_tipo = serie_tipo[serie_tipo["Prima anual"] > 0]

estadisticos = serie_tipo.groupby("tipo_embarcacion")["Prima anual"].agg([
    "count", "mean", "std", "min", lambda x: x.quantile(0.25), "median",
    lambda x: x.quantile(0.75), "max",
])
estadisticos.columns = [
    "N", "Media", "Desv. estándar", "Mínimo", "Q1", "Mediana", "Q3", "Máximo"
]
q1 = estadisticos["Q1"]
q3 = estadisticos["Q3"]
iqr = q3 - q1
estadisticos["Límite inferior"] = q1 - 1.5 * iqr
estadisticos["Límite superior"] = q3 + 1.5 * iqr
estadisticos = estadisticos.sort_values("Media", ascending=False)

display(estadisticos.style.format("{:,.2f}"))

atipicos = serie_tipo.merge(
    estadisticos[["Límite inferior", "Límite superior"]],
    left_on="tipo_embarcacion",
    right_index=True,
).query(
    "`Prima anual` < `Límite inferior` or `Prima anual` > `Límite superior`"
).drop(columns=["Límite inferior", "Límite superior"])

if atipicos.empty:
    print("No se identificaron valores atípicos por el criterio 1.5 × IQR.")
else:
    display(atipicos.sort_values("Prima anual", ascending=False).style.format(
        {"Prima anual": "{:,.2f}"}
    ))


,N,Media,Desv. estándar,Mínimo,Q1,Mediana,Q3,Máximo,Límite inferior,Límite superior
tipo_embarcacion,,,,,,,,,,
buque-tanque petrolero y productos derivados,11.00,"1,139,324,406.51","1,422,475,241.32","18,952,760.16","207,103,924.65","617,161,944.31","1,561,787,144.14","4,760,401,094.00","-1,824,920,904.59","3,593,811,973.39"
abastecedor,11.00,"807,562,189.06","656,542,547.49","244,938,831.00","451,208,291.60","600,255,735.80","919,945,570.64","2,577,921,400.07","-251,897,626.97","1,623,051,489.21"
lancha,11.00,"195,367,593.63","46,140,324.39","97,962,926.02","173,493,151.14","198,904,512.45","232,879,012.98","249,875,524.96","84,414,358.38","321,957,805.73"
pesca multiple,11.00,"192,976,465.02","86,180,919.89","98,652,019.49","112,454,085.77","175,462,057.59","276,407,198.68","319,546,235.17","-133,475,583.60","522,336,868.05"
yate,11.00,"173,624,942.50","48,617,570.54","100,911,677.99","134,839,949.23","180,947,676.39","210,134,240.97","247,629,934.00","21,898,511.62","323,075,678.58"
otro tipo de embarcacion,11.00,"157,924,163.19","209,100,293.48","1,898,260.06","14,873,136.70","75,959,493.64","171,096,825.61","645,238,938.24","-219,462,396.67","405,432,358.98"
carguero general,11.00,"122,870,763.31","148,102,345.87","29,108,499.00","56,629,065.55","85,009,819.43","109,753,143.73","561,350,498.59","-23,057,051.72","189,439,260.99"
buque-tanque,11.00,"98,026,851.77","59,863,802.01","22,559,716.18","61,907,135.72","96,295,496.13","109,667,515.01","245,248,642.00","-9,733,433.21","181,308,083.95"
militar u oficial,10.00,"92,747,902.25","63,279,584.85","3,333,337.83","51,221,087.37","94,995,453.44","127,239,464.11","196,576,688.23","-62,806,477.74","241,267,029.22"


,año,tipo_embarcacion,Prima anual
318,2025,buque-tanque petrolero y productos derivados,"4,760,401,094.00"
124,2019,abastecedor,"2,577,921,400.07"
116,2018,otro tipo de embarcacion,"645,238,938.24"
261,2023,carguero general,"561,350,498.59"
85,2017,otro tipo de embarcacion,"457,859,530.64"
317,2025,buque-tanque,"245,248,642.00"
126,2019,atunero,"202,739,817.27"
65,2017,barcaza o chalan,"120,767,660.37"
97,2018,barco-escuela,"77,955,221.33"
296,2024,embarcacion menor utilizada para vigilancia,"71,870,949.94"


#### Distribución anual y cambios principales

In [36]:
from IPython.display import clear_output
clear_output(wait=True)

participacion = anual_tipo.div(anual_tipo.sum(axis=1), axis=0).mul(100).fillna(0)
cambios_pp = participacion.diff()

ultimo_anio = anual_tipo.index[-1]
ranking_acumulado = (
    anual_tipo.sum()
    .mul(100 / anual_tipo.sum().sum())
    .sort_values(ascending=False)
)
ranking_cierre = participacion.loc[ultimo_anio].sort_values(ascending=False)
ranking = pd.DataFrame({
    "Participación acumulada (%)": ranking_acumulado,
    f"Participación {ultimo_anio} (%)": ranking_cierre,
}).sort_values("Participación acumulada (%)", ascending=False)

display(participacion.style.format("{:.2f}%"))
display(cambios_pp.style.format("{:+.2f}"))
display(ranking.style.format("{:.2f}%"))

print(f"Tipo líder acumulado: {ranking.index[0]} ({ranking.iloc[0, 0]:.2f}%).")
print(f"Tipo rezagado acumulado: {ranking.index[-1]} ({ranking.iloc[-1, 0]:.2f}%).")
print(f"Tipo líder al cierre: {ranking_cierre.index[0]} ({ranking_cierre.iloc[0]:.2f}%).")

cambios_validos = cambios_pp.stack()
cambio_max = cambios_validos.idxmax()
cambio_min = cambios_validos.idxmin()
print(f"Mayor aumento: {cambios_validos.max():+.2f} pp ({cambio_max[1]} en {cambio_max[0]}).")
print(f"Mayor caída: {cambios_validos.min():+.2f} pp ({cambio_min[1]} en {cambio_min[0]}).")


tipo_embarcacion,abastecedor,artefactos navales,atunero,barcaza o chalan,barco-escuela,barco-hidrografico,buque transportador de autos,buque-tanque,buque-tanque petrolero y productos derivados,buque-tanque quimico,buque-tanque-minerales,camaronero,carguero a granel,carguero general,carguero tipo roll on - roll off,dragas,embarcacion de investigacion,embarcacion menor utilizada para vigilancia,escamero,ferry-transbordador,jet sky o moto acuatica,lancha,militar u oficial,otro tipo de embarcacion,pesca multiple,porta contenedor,remolcador,sardinero o anchovetero,tiburoneros,velero,yate
año,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015,41.17%,0.00%,1.83%,0.66%,0.00%,0.00%,0.00%,3.78%,18.82%,0.00%,0.00%,0.67%,0.03%,2.78%,0.00%,0.28%,0.22%,0.00%,0.01%,1.79%,0.30%,6.41%,2.24%,7.30%,6.18%,0.00%,0.39%,1.10%,0.04%,0.44%,3.56%
2016,42.67%,0.09%,1.06%,0.52%,0.00%,0.01%,0.00%,3.77%,22.77%,0.36%,0.00%,0.80%,0.02%,4.48%,0.00%,0.13%,0.12%,0.00%,0.01%,1.10%,0.13%,9.21%,0.12%,0.70%,4.19%,0.00%,0.53%,1.69%,0.00%,1.10%,4.42%
2017,20.08%,0.01%,2.07%,3.92%,0.03%,0.00%,0.01%,2.46%,32.61%,0.08%,0.00%,0.39%,0.02%,3.52%,0.00%,0.09%,0.11%,0.01%,0.01%,0.81%,1.56%,6.56%,0.00%,14.86%,3.62%,0.00%,0.54%,1.30%,0.00%,1.61%,3.71%
2018,16.08%,0.51%,0.93%,0.72%,2.88%,0.00%,0.00%,1.51%,29.07%,0.00%,0.03%,0.94%,0.01%,4.10%,0.00%,0.06%,0.64%,0.00%,0.04%,1.06%,0.15%,3.62%,2.64%,23.81%,3.95%,0.00%,0.51%,0.60%,0.00%,0.42%,5.74%
2019,70.40%,0.00%,5.54%,0.70%,0.14%,0.00%,0.00%,1.32%,1.25%,0.00%,0.01%,0.81%,0.01%,1.58%,0.00%,0.13%,0.33%,0.71%,0.04%,0.40%,0.13%,6.12%,0.39%,1.63%,2.69%,0.00%,0.47%,0.73%,0.00%,0.37%,4.09%
2020,37.37%,0.04%,1.94%,0.50%,0.12%,0.00%,0.01%,1.23%,9.89%,0.00%,0.03%,1.62%,0.61%,3.03%,0.00%,0.30%,0.37%,0.01%,0.05%,1.22%,0.27%,9.96%,7.10%,0.10%,6.99%,0.00%,3.86%,0.58%,0.00%,0.85%,11.93%
2021,30.10%,0.12%,1.52%,0.11%,0.06%,0.00%,0.00%,7.05%,0.95%,0.00%,0.00%,1.11%,0.06%,4.41%,0.00%,0.22%,0.63%,0.01%,0.05%,1.37%,0.26%,12.53%,5.98%,6.78%,14.54%,0.00%,1.47%,0.75%,0.00%,0.85%,9.07%
2022,17.80%,0.19%,1.81%,0.31%,0.00%,0.00%,0.00%,5.66%,11.78%,0.00%,0.00%,1.22%,0.06%,4.29%,0.05%,0.26%,0.42%,0.01%,0.04%,0.64%,0.30%,12.20%,5.99%,6.11%,16.14%,0.00%,2.10%,0.33%,0.00%,1.86%,10.44%
2023,12.45%,0.05%,0.79%,0.12%,0.03%,0.00%,0.00%,1.88%,48.01%,0.00%,0.00%,0.35%,0.07%,12.08%,0.01%,0.09%,0.41%,0.01%,0.03%,0.41%,0.13%,4.28%,4.23%,1.63%,5.88%,0.00%,1.81%,0.15%,0.00%,0.76%,4.36%


tipo_embarcacion,abastecedor,artefactos navales,atunero,barcaza o chalan,barco-escuela,barco-hidrografico,buque transportador de autos,buque-tanque,buque-tanque petrolero y productos derivados,buque-tanque quimico,buque-tanque-minerales,camaronero,carguero a granel,carguero general,carguero tipo roll on - roll off,dragas,embarcacion de investigacion,embarcacion menor utilizada para vigilancia,escamero,ferry-transbordador,jet sky o moto acuatica,lancha,militar u oficial,otro tipo de embarcacion,pesca multiple,porta contenedor,remolcador,sardinero o anchovetero,tiburoneros,velero,yate
año,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan
2016,+1.50,+0.08,-0.76,-0.14,-0.00,+0.01,-0.00,-0.00,+3.95,+0.36,+0.00,+0.13,-0.01,+1.70,+0.00,-0.15,-0.10,+0.00,-0.00,-0.69,-0.17,+2.80,-2.12,-6.60,-1.99,+0.00,+0.13,+0.59,-0.04,+0.66,+0.86
2017,-22.59,-0.08,+1.01,+3.40,+0.03,-0.01,+0.01,-1.32,+9.84,-0.28,+0.00,-0.41,-0.00,-0.95,+0.00,-0.04,-0.01,+0.00,+0.00,-0.28,+1.44,-2.66,-0.12,+14.17,-0.57,+0.00,+0.01,-0.38,+0.00,+0.51,-0.71
2018,-4.01,+0.50,-1.14,-3.20,+2.84,+0.00,-0.01,-0.94,-3.55,-0.08,+0.03,+0.54,-0.00,+0.57,+0.00,-0.03,+0.53,-0.00,+0.03,+0.25,-1.41,-2.94,+2.64,+8.95,+0.34,+0.00,-0.03,-0.70,+0.00,-1.19,+2.03
2019,+54.33,-0.51,+4.61,-0.02,-2.74,-0.00,-0.00,-0.20,-27.81,+0.00,-0.01,-0.12,-0.00,-2.52,+0.00,+0.08,-0.32,+0.71,-0.00,-0.66,-0.03,+2.51,-2.24,-22.18,-1.26,+0.00,-0.04,+0.13,+0.00,-0.05,-1.64
2020,-33.03,+0.04,-3.60,-0.20,-0.02,-0.00,+0.01,-0.08,+8.64,+0.00,+0.02,+0.81,+0.60,+1.45,+0.00,+0.17,+0.04,-0.70,+0.02,+0.82,+0.14,+3.84,+6.71,-1.53,+4.30,-0.00,+3.39,-0.15,+0.00,+0.48,+7.84
2021,-7.27,+0.08,-0.41,-0.39,-0.06,+0.00,-0.01,+5.81,-8.94,+0.00,-0.03,-0.51,-0.55,+1.38,+0.00,-0.09,+0.26,+0.00,-0.00,+0.14,-0.01,+2.57,-1.12,+6.67,+7.55,+0.00,-2.39,+0.17,-0.00,-0.00,-2.86
2022,-12.30,+0.08,+0.28,+0.20,-0.06,+0.00,+0.00,-1.38,+10.83,+0.00,+0.00,+0.10,+0.01,-0.11,+0.04,+0.04,-0.21,-0.00,-0.01,-0.72,+0.04,-0.33,+0.01,-0.67,+1.60,+0.00,+0.63,-0.42,-0.00,+1.01,+1.36
2023,-5.35,-0.15,-1.02,-0.19,+0.02,+0.00,+0.00,-3.79,+36.23,+0.00,+0.00,-0.86,+0.01,+7.79,-0.04,-0.17,-0.01,+0.00,-0.01,-0.24,-0.18,-7.92,-1.76,-4.47,-10.26,+0.00,-0.29,-0.18,+0.00,-1.10,-6.08


,Participación acumulada (%),Participación 2025 (%)
tipo_embarcacion,,
buque-tanque petrolero y productos derivados,35.40%,77.04%
abastecedor,25.09%,3.96%
lancha,6.07%,2.67%
pesca multiple,6.00%,4.53%
yate,5.40%,4.01%
otro tipo de embarcacion,4.91%,0.18%
carguero general,3.82%,0.47%
buque-tanque,3.05%,3.97%
militar u oficial,2.62%,0.76%


Tipo líder acumulado: buque-tanque petrolero y productos derivados (35.40%).
Tipo rezagado acumulado: barco-hidrografico (0.00%).
Tipo líder al cierre: buque-tanque petrolero y productos derivados (77.04%).
Mayor aumento: +54.33 pp (abastecedor en 2019).
Mayor caída: -33.03 pp (abastecedor en 2020).


#### Caja y bigotes de la prima por tipo

In [37]:
from IPython.display import clear_output
clear_output(wait=True)

fig = go.Figure()
for tipo in estadisticos.index:
    valores = serie_tipo.loc[serie_tipo["tipo_embarcacion"] == tipo, "Prima anual"]
    fig.add_box(
        y=valores,
        x=[tipo] * len(valores),
        name=tipo,
        boxpoints="all",
        jitter=0.25,
        pointpos=0,
        marker=dict(size=5, opacity=0.65, color=COLORES["naranja"]),
        line=dict(color=COLORES["navy"], width=1),
        showlegend=False,
    )

fig.update_layout(
    title="Distribución anual de la Prima Emitida por tipo",
    template="reservas",
    height=600,
    yaxis=dict(title="Prima emitida anual (pesos constantes)", tickformat="~s"),
    xaxis=dict(title="Tipo de embarcación", tickangle=-45),
)
fig.show()


#### Composición 100% por tipo

In [38]:
from IPython.display import clear_output
clear_output(wait=True)

fig = go.Figure()

# Ordenar los tipos de mayor a menor participación dentro de cada año.
# Cada posición ocupa la misma altura relativa en todas las barras.
for posicion in range(participacion.shape[1]):
    valores = []
    tipos_posicion = []
    colores_posicion = []

    for año in participacion.index:
        fila = participacion.loc[año].sort_values(ascending=False)
        tipo = fila.index[posicion] if posicion < len(fila) else None
        valor = fila.iloc[posicion] if posicion < len(fila) else 0
        valores.append(valor)
        tipos_posicion.append(tipo if tipo is not None else "Sin participación")
        if tipo is not None:
            indice_tipo = list(participacion.columns).index(tipo)
            colores_posicion.append(COLORES_TIPOS[indice_tipo % len(COLORES_TIPOS)])
        else:
            colores_posicion.append("rgba(0,0,0,0)")

    fig.add_bar(
        x=participacion.index,
        y=valores,
        customdata=tipos_posicion,
        hovertemplate="Año %{x}<br>%{customdata}<br>Participación: %{y:.2f}%<extra></extra>",
        name=f"Posición {posicion + 1}",
        marker_color=colores_posicion,
        showlegend=False,
    )

# Leyenda separada con los tipos reales y sus colores habituales.
for i, tipo in enumerate(participacion.columns):
    fig.add_scatter(
        x=[None],
        y=[None],
        mode="markers",
        name=tipo,
        marker=dict(size=10, color=COLORES_TIPOS[i % len(COLORES_TIPOS)]),
        showlegend=True,
    )

fig.update_layout(
    title="Composición anual del mercado por tipo",
    template="reservas",
    barmode="stack",
    showlegend=True,
    barnorm="percent",
    height=700,
    margin=dict(l=65, r=280, t=90, b=65),
    legend=dict(
        orientation="v",
        x=1.02,
        xanchor="left",
        y=1,
        yanchor="top",
        font=dict(size=11),
        bgcolor="rgba(242,242,242,0.95)",
        borderwidth=0,
    ),
    xaxis=dict(title="Año", dtick=1),
    yaxis=dict(title="Participación", ticksuffix="%", range=[0, 100]),
)
fig.show()


#### Ranking acumulado de participación

In [39]:
from IPython.display import clear_output
clear_output(wait=True)

ranking_grafico = ranking_acumulado.sort_values(ascending=True)
fig = go.Figure(go.Bar(
    x=ranking_grafico.values,
    y=ranking_grafico.index,
    orientation="h",
    marker_color=COLORES["azul"],
    text=ranking_grafico.map(lambda valor: f"{valor:.1f}%"),
    textposition="outside",
))
fig.update_layout(
    title="Participación acumulada por tipo",
    template="reservas",
    height=max(520, 18 * len(ranking_grafico)),
    xaxis=dict(title="Participación acumulada", ticksuffix="%", range=[0, ranking_grafico.max() * 1.12]),
    yaxis=dict(title=""),
)
fig.show()


## 2.2. Monto del sinestro

### 3a. Mercado total: monto anual de siniestros

Se calcula el monto total sumando los cuatro desarrollos ajustados de cada registro y agrupando por el año de origen `año`. Las cifras se mantienen en pesos constantes de diciembre de 2025.


In [40]:
from IPython.display import clear_output
clear_output(wait=True)

import numpy as np
import plotly.graph_objects as go

# Sumar los pagos ajustados de los cuatro desarrollos por registro
columnas_pagos_ajustados = [
    f"{desarrollo}_monto pagado_ajustada"
    for desarrollo in range(4)
]

siniestros_analisis = sinestralidad_ajustada.copy()
siniestros_analisis["monto_siniestro"] = siniestros_analisis[
    columnas_pagos_ajustados
].sum(axis=1, min_count=1)

siniestros_mercado = (
    siniestros_analisis
    .groupby("año", as_index=True)["monto_siniestro"]
    .sum()
    .sort_index()
    .to_frame()
)
siniestros_mercado["variacion_monetaria"] = siniestros_mercado["monto_siniestro"].diff()
siniestros_mercado["crecimiento_real"] = siniestros_mercado["monto_siniestro"].pct_change() * 100

primer_anio = siniestros_mercado.index.min()
ultimo_anio = siniestros_mercado.index.max()
periodos = ultimo_anio - primer_anio
cagr_siniestros = (
    siniestros_mercado.loc[ultimo_anio, "monto_siniestro"]
    / siniestros_mercado.loc[primer_anio, "monto_siniestro"]
) ** (1 / periodos) - 1
cagr_siniestros *= 100

promedio_siniestros = siniestros_mercado["monto_siniestro"].mean()
desviacion_siniestros = siniestros_mercado["monto_siniestro"].std()
volatilidad_crecimiento = siniestros_mercado["crecimiento_real"].std()

coeficiente_siniestros = np.polyfit(
    siniestros_mercado.index.astype(float),
    siniestros_mercado["monto_siniestro"].to_numpy(),
    1,
)
pendiente_siniestros = coeficiente_siniestros[0]
tendencia_siniestros = np.polyval(
    coeficiente_siniestros,
    siniestros_mercado.index.astype(float),
)

tabla_siniestros = siniestros_mercado.rename(columns={
    "monto_siniestro": "Monto anual",
    "variacion_monetaria": "Variación monetaria",
    "crecimiento_real": "Crecimiento real (%)",
})

display(tabla_siniestros.style.format({
    "Monto anual": "{:,.2f}",
    "Variación monetaria": "{:,.2f}",
    "Crecimiento real (%)": "{:.2f}",
}))
print(f"TCAC/CAGR: {cagr_siniestros:.2f}%")
print(f"Promedio anual: {promedio_siniestros:,.2f}")
print(f"Desviación estándar del monto anual: {desviacion_siniestros:,.2f}")
print(f"Desviación estándar del crecimiento anual: {volatilidad_crecimiento:.2f}%")
print(f"Pendiente lineal: {pendiente_siniestros:,.2f} pesos/año")

anio_pico = siniestros_mercado["monto_siniestro"].idxmax()
anio_valle = siniestros_mercado["monto_siniestro"].idxmin()
print(
    f"Pico: {siniestros_mercado.loc[anio_pico, 'monto_siniestro']:,.2f} "
    f"pesos en {anio_pico}."
)
print(
    f"Valle: {siniestros_mercado.loc[anio_valle, 'monto_siniestro']:,.2f} "
    f"pesos en {anio_valle}."
)

colores_barras = [
    COLORES["naranja"] if year == anio_pico else COLORES["azul"]
    for year in siniestros_mercado.index
]

fig = go.Figure()
fig.add_bar(
    x=siniestros_mercado.index,
    y=siniestros_mercado["monto_siniestro"],
    name="Monto anual",
    marker_color=colores_barras,
)
fig.add_scatter(
    x=siniestros_mercado.index,
    y=tendencia_siniestros,
    mode="lines",
    name="Tendencia lineal",
    line=dict(color=COLORES["navy"], dash="dot", width=3),
)
fig.add_scatter(
    x=siniestros_mercado.index,
    y=siniestros_mercado["crecimiento_real"],
    mode="lines+markers",
    name="Crecimiento real",
    yaxis="y2",
    line=dict(color=COLORES["verde"], width=2),
)
fig.add_annotation(
    x=anio_pico,
    y=siniestros_mercado.loc[anio_pico, "monto_siniestro"],
    text=f"Pico {anio_pico}",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-55,
)
fig.update_layout(
    title=f"Monto anual de siniestros — CAGR {cagr_siniestros:.2f}%",
    template="reservas",
    height=520,
    legend=dict(orientation="h", y=1.02, x=0),
    yaxis=dict(title="Monto (pesos constantes)", tickformat="~s"),
    yaxis2=dict(title="Crecimiento (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis=dict(title="Año", dtick=1),
)
fig.show()


,Monto anual,Variación monetaria,Crecimiento real (%)
año,,,
2015,"4,285,569,030.11",nan,nan
2016,"1,478,908,507.45","-2,806,660,522.65",-65.49
2017,"1,021,553,452.08","-457,355,055.37",-30.93
2018,"959,508,132.92","-62,045,319.15",-6.07
2019,"1,390,271,331.35","430,763,198.43",44.89
2020,"1,070,848,012.55","-319,423,318.80",-22.98
2021,"1,016,902,419.30","-53,945,593.26",-5.04
2022,"1,322,134,076.59","305,231,657.29",30.02
2023,"2,038,944,451.28","716,810,374.70",54.22


TCAC/CAGR: -6.85%
Promedio anual: 1,769,102,629.64
Desviación estándar del monto anual: 1,010,557,814.90
Desviación estándar del crecimiento anual: 38.77%
Pendiente lineal: -21,177,514.67 pesos/año
Pico: 4,285,569,030.11 pesos en 2015.
Valle: 959,508,132.92 pesos en 2018.


### 3b. Distribución del monto anual de siniestros por causa

Se presentan tablas completas con montos absolutos, participación porcentual, cambios de participación y ranking acumulado por causa.


In [41]:
from IPython.display import clear_output
clear_output(wait=True)

# Pivotes completos por causa del siniestro
causa_monto = (
    siniestros_analisis
    .groupby(["año", "causa_siniestro"])["monto_siniestro"]
    .sum()
    .unstack(fill_value=0)
)
causa_monto.loc["Total"] = causa_monto.sum(axis=0)
causa_monto["Total"] = causa_monto.sum(axis=1)

causa_porcentaje = causa_monto.drop(index="Total").drop(columns="Total")
causa_porcentaje = causa_porcentaje.div(
    causa_porcentaje.sum(axis=1), axis=0
).mul(100)
causa_porcentaje["Total"] = 100.0
acumulado_causas = causa_porcentaje.drop(columns="Total").sum(axis=0)
monto_total_acumulado = causa_monto.drop(index="Total").drop(columns="Total").to_numpy().sum()
causa_porcentaje.loc["Total"] = (
    causa_monto.loc["Total"].drop(index="Total", errors="ignore")
    if False else
    causa_monto.drop(index="Total").drop(columns="Total").sum(axis=0)
)
causa_porcentaje.loc["Total"] = causa_porcentaje.loc["Total"] / monto_total_acumulado * 100
causa_porcentaje.loc["Total", "Total"] = 100.0

cambios_causa_pp = causa_porcentaje.drop(index="Total").drop(columns="Total").diff()

print("Monto absoluto anual por causa del siniestro")
display(causa_monto.style.format("{:,.2f}"))

print("Distribución porcentual anual por causa del siniestro")
display(causa_porcentaje.style.format("{:.2f}%"))

print("Cambios interanuales en puntos porcentuales")
display(cambios_causa_pp.style.format("{:+.2f}"))

ranking_causas = pd.DataFrame({
    "Monto acumulado": causa_monto.drop(index="Total").drop(columns="Total").sum(axis=0),
    "Participación acumulada (%)": (
        causa_monto.drop(index="Total").drop(columns="Total").sum(axis=0)
        / causa_monto.drop(index="Total").drop(columns="Total").to_numpy().sum()
        * 100
    ),
    "Promedio anual": causa_monto.drop(index="Total").drop(columns="Total").mean(axis=0),
}).sort_values("Monto acumulado", ascending=False)

print("Ranking acumulado por causa")
display(ranking_causas.style.format({
    "Monto acumulado": "{:,.2f}",
    "Participación acumulada (%)": "{:.2f}%",
    "Promedio anual": "{:,.2f}",
}))

causa_principal = ranking_causas.index[0]
causa_marginal = ranking_causas.index[-1]
print(f"Causa principal: {causa_principal} ({ranking_causas.iloc[0]['Participación acumulada (%)']:.2f}% del total).")
print(f"Causa marginal: {causa_marginal} ({ranking_causas.iloc[-1]['Participación acumulada (%)']:.2f}% del total).")

cambios_validos = cambios_causa_pp.stack()
if cambios_validos.notna().any():
    cambio_max = cambios_validos.idxmax()
    cambio_min = cambios_validos.idxmin()
    print(f"Mayor aumento de participación: {cambios_validos.max():+.2f} pp ({cambio_max[1]} en {cambio_max[0]}).")
    print(f"Mayor caída de participación: {cambios_validos.min():+.2f} pp ({cambio_min[1]} en {cambio_min[0]}).")


Monto absoluto anual por causa del siniestro


causa_siniestro,Abordaje,Averia gruesa,Averia particular,Barateria del capitan o tripulacion,Colision,Encallamiento,Fenomenos de la naturaleza,Guerra y Secuestro,Huelgas y Alborotos populares,Incendio,Incendio,Motin,Motin,Negligencia de la tripulacion,Otra causa,Rayo o explosion,Robo total,Sabotaje,Terrorismo,Varadura,Total
año,,,,,,,,,,,,,,,,,,,,,
2015,"1,744,331.86","54,505,389.21","222,598,075.45","48,150.89","8,724,486.21","3,492,613.77","3,787,488,033.80",0.00,0.00,"47,682,859.58",0.00,"1,983,277.12",0.00,"85,664,362.98","2,084,892.77","18,623,931.05","14,094,145.30","4,403,610.15",0.00,"32,430,869.96","4,285,569,030.11"
2016,"1,793,684.08","156,051,699.56","317,517,324.21","1,780,373.40","24,712,974.90","35,760,692.60","63,858,708.75",0.00,0.00,0.00,"419,653,190.69",0.00,0.00,"107,960,563.15","10,293,178.83","116,188,030.70","27,739,985.77","70,523,276.69",0.00,"125,074,824.15","1,478,908,507.45"
2017,"1,378,650.82","73,022,876.71","117,445,092.61","5,072,549.62","42,023,335.65","69,958,769.06","35,273,321.62",0.00,0.00,0.00,"194,205,984.92",0.00,"1,371.14","456,321,671.05","18,815.21","651,174.08","8,026,199.03","1,182,896.58",0.00,"16,970,743.96","1,021,553,452.08"
2018,"478,790.57","307,045,452.50","247,600,289.27","68,449.67","3,710,876.96","58,520,270.61","23,047,831.67",0.00,"1,595,255.70",0.00,"140,300,752.32",0.00,0.00,"142,449,433.59","2,712,206.30","20,378,020.40","3,408,950.94",0.00,0.00,"8,191,552.42","959,508,132.92"
2019,"279,481,069.41","151,698,660.04","609,199,414.10","975,485.26","4,110,776.16","10,852,029.57","10,316,899.82",0.00,0.00,0.00,"63,960,406.35",0.00,0.00,"23,327,421.49","255,783.41","13,629,873.97","222,383,662.79",0.00,0.00,"79,848.97","1,390,271,331.35"
2020,"2,547,204.39","193,490,917.43","179,621,215.78","41,358.36","17,977,396.38","31,199,845.81","36,668,771.25",0.00,0.00,0.00,"341,797,999.81",0.00,0.00,"26,106,856.61","46,693,443.89","38,084,130.52","84,422,802.82",0.00,0.00,"72,196,069.52","1,070,848,012.55"
2021,"36,939,236.63","101,800,819.91","95,699,518.00","1,182,324.29","29,452,674.91","24,011,521.36","126,602,244.13","35,069.45",0.00,0.00,"182,757,212.98",0.00,"254,863.22","72,539,771.28","821,148.23","144,057,200.27","194,035,217.07",0.00,"1,927,543.02","4,786,054.56","1,016,902,419.30"
2022,"18,416,174.63","336,948,188.32","232,255,260.36",0.00,"16,118,098.87","24,672,824.47","11,221,918.07",0.00,0.00,0.00,"518,753,788.80",0.00,0.00,"5,756,521.46",0.00,"92,609,362.18","10,308,267.94","60,405.90",0.00,"55,013,265.59","1,322,134,076.59"
2023,"29,720,035.59","310,231,690.56","170,027,129.08",0.00,"37,790,822.29","45,886,556.33","552,250,338.17",0.00,0.00,0.00,"168,206,984.32",0.00,"233,299.81","165,403,901.63","41,800,625.55","92,757,705.89","47,844,287.37","1,807,198.37",0.00,"374,983,876.31","2,038,944,451.28"


Distribución porcentual anual por causa del siniestro


causa_siniestro,Abordaje,Averia gruesa,Averia particular,Barateria del capitan o tripulacion,Colision,Encallamiento,Fenomenos de la naturaleza,Guerra y Secuestro,Huelgas y Alborotos populares,Incendio,Incendio,Motin,Motin,Negligencia de la tripulacion,Otra causa,Rayo o explosion,Robo total,Sabotaje,Terrorismo,Varadura,Total
año,,,,,,,,,,,,,,,,,,,,,
2015,0.04%,1.27%,5.19%,0.00%,0.20%,0.08%,88.38%,0.00%,0.00%,1.11%,0.00%,0.05%,0.00%,2.00%,0.05%,0.43%,0.33%,0.10%,0.00%,0.76%,100.00%
2016,0.12%,10.55%,21.47%,0.12%,1.67%,2.42%,4.32%,0.00%,0.00%,0.00%,28.38%,0.00%,0.00%,7.30%,0.70%,7.86%,1.88%,4.77%,0.00%,8.46%,100.00%
2017,0.13%,7.15%,11.50%,0.50%,4.11%,6.85%,3.45%,0.00%,0.00%,0.00%,19.01%,0.00%,0.00%,44.67%,0.00%,0.06%,0.79%,0.12%,0.00%,1.66%,100.00%
2018,0.05%,32.00%,25.80%,0.01%,0.39%,6.10%,2.40%,0.00%,0.17%,0.00%,14.62%,0.00%,0.00%,14.85%,0.28%,2.12%,0.36%,0.00%,0.00%,0.85%,100.00%
2019,20.10%,10.91%,43.82%,0.07%,0.30%,0.78%,0.74%,0.00%,0.00%,0.00%,4.60%,0.00%,0.00%,1.68%,0.02%,0.98%,16.00%,0.00%,0.00%,0.01%,100.00%
2020,0.24%,18.07%,16.77%,0.00%,1.68%,2.91%,3.42%,0.00%,0.00%,0.00%,31.92%,0.00%,0.00%,2.44%,4.36%,3.56%,7.88%,0.00%,0.00%,6.74%,100.00%
2021,3.63%,10.01%,9.41%,0.12%,2.90%,2.36%,12.45%,0.00%,0.00%,0.00%,17.97%,0.00%,0.03%,7.13%,0.08%,14.17%,19.08%,0.00%,0.19%,0.47%,100.00%
2022,1.39%,25.49%,17.57%,0.00%,1.22%,1.87%,0.85%,0.00%,0.00%,0.00%,39.24%,0.00%,0.00%,0.44%,0.00%,7.00%,0.78%,0.00%,0.00%,4.16%,100.00%
2023,1.46%,15.22%,8.34%,0.00%,1.85%,2.25%,27.09%,0.00%,0.00%,0.00%,8.25%,0.00%,0.01%,8.11%,2.05%,4.55%,2.35%,0.09%,0.00%,18.39%,100.00%


Cambios interanuales en puntos porcentuales


causa_siniestro,Abordaje,Averia gruesa,Averia particular,Barateria del capitan o tripulacion,Colision,Encallamiento,Fenomenos de la naturaleza,Guerra y Secuestro,Huelgas y Alborotos populares,Incendio,Incendio,Motin,Motin,Negligencia de la tripulacion,Otra causa,Rayo o explosion,Robo total,Sabotaje,Terrorismo,Varadura
año,,,,,,,,,,,,,,,,,,,,
2015,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan
2016,+0.08,+9.28,+16.28,+0.12,+1.47,+2.34,-84.06,+0.00,+0.00,-1.11,+28.38,-0.05,+0.00,+5.30,+0.65,+7.42,+1.55,+4.67,+0.00,+7.70
2017,+0.01,-3.40,-9.97,+0.38,+2.44,+4.43,-0.87,+0.00,+0.00,+0.00,-9.37,+0.00,+0.00,+37.37,-0.69,-7.79,-1.09,-4.65,+0.00,-6.80
2018,-0.09,+24.85,+14.31,-0.49,-3.73,-0.75,-1.05,+0.00,+0.17,+0.00,-4.39,+0.00,-0.00,-29.82,+0.28,+2.06,-0.43,-0.12,+0.00,-0.81
2019,+20.05,-21.09,+18.01,+0.06,-0.09,-5.32,-1.66,+0.00,-0.17,+0.00,-10.02,+0.00,+0.00,-13.17,-0.26,-1.14,+15.64,+0.00,+0.00,-0.85
2020,-19.86,+7.16,-27.05,-0.07,+1.38,+2.13,+2.68,+0.00,+0.00,+0.00,+27.32,+0.00,+0.00,+0.76,+4.34,+2.58,-8.11,+0.00,+0.00,+6.74
2021,+3.39,-8.06,-7.36,+0.11,+1.22,-0.55,+9.03,+0.00,+0.00,+0.00,-13.95,+0.00,+0.03,+4.70,-4.28,+10.61,+11.20,+0.00,+0.19,-6.27
2022,-2.24,+15.47,+8.16,-0.12,-1.68,-0.50,-11.60,-0.00,+0.00,+0.00,+21.26,+0.00,-0.03,-6.70,-0.08,-7.16,-18.30,+0.00,-0.19,+3.69
2023,+0.06,-10.27,-9.23,+0.00,+0.63,+0.38,+26.24,+0.00,+0.00,+0.00,-30.99,+0.00,+0.01,+7.68,+2.05,-2.46,+1.57,+0.08,+0.00,+14.23


Ranking acumulado por causa


,Monto acumulado,Participación acumulada (%),Promedio anual
causa_siniestro,,,
Fenomenos de la naturaleza,"5,861,049,533.90",30.12%,"532,822,684.90"
Incendio,"3,338,554,823.17",17.16%,"303,504,983.92"
Averia particular,"2,529,384,679.21",13.00%,"229,944,061.75"
Averia gruesa,"2,045,173,715.02",10.51%,"185,924,883.18"
Varadura,"1,504,271,031.17",7.73%,"136,751,911.92"
Negligencia de la tripulacion,"1,230,163,453.77",6.32%,"111,833,041.25"
Rayo o explosion,"737,721,245.02",3.79%,"67,065,567.73"
Robo total,"733,614,368.44",3.77%,"66,692,215.31"
Colision,"462,729,510.43",2.38%,"42,066,319.13"


Causa principal: Fenomenos de la naturaleza (30.12% del total).
Causa marginal: Motin  (0.00% del total).
Mayor aumento de participación: +37.37 pp (Negligencia de la tripulacion en 2017).
Mayor caída de participación: -84.06 pp (Fenomenos de la naturaleza en 2016).


#### Distribución porcentual anual por causa

Las causas marginales se agrupan como «Otras causas» únicamente en esta gráfica; las tablas conservan todas las categorías registradas.


In [42]:
from IPython.display import clear_output
clear_output(wait=True)

participacion_causas = causa_porcentaje.drop(index="Total").drop(columns="Total")
participacion_grafica = participacion_causas.copy()
causas_marginales = []
for causa in participacion_grafica.columns:
    if participacion_grafica[causa].max() < 2:
        causas_marginales.append(causa)

if causas_marginales:
    participacion_grafica["Otras causas"] = participacion_grafica[causas_marginales].sum(axis=1)
    participacion_grafica = participacion_grafica.drop(columns=causas_marginales)

fig = go.Figure()

# Ordenar causas de mayor a menor participación dentro de cada año.
for posicion in range(participacion_grafica.shape[1]):
    valores = []
    causas_posicion = []
    colores_posicion = []

    for año in participacion_grafica.index:
        ranking = participacion_grafica.loc[año].sort_values(ascending=False)
        causa = ranking.index[posicion] if posicion < len(ranking) else None
        valor = ranking.iloc[posicion] if posicion < len(ranking) else 0

        valores.append(valor)
        causas_posicion.append(causa if causa is not None else "Sin participación")

        if causa is not None:
            indice_causa = list(participacion_grafica.columns).index(causa)
            colores_posicion.append(COLORES_TIPOS[indice_causa % len(COLORES_TIPOS)])
        else:
            colores_posicion.append("rgba(0,0,0,0)")

    fig.add_bar(
        x=participacion_grafica.index,
        y=valores,
        customdata=causas_posicion,
        hovertemplate="Año %{x}<br>%{customdata}<br>Participación: %{y:.2f}%<extra></extra>",
        marker_color=colores_posicion,
        name=f"Posición {posicion + 1}",
        showlegend=False,
    )

# Leyenda separada con nombres y colores reales de las causas.
for i, causa in enumerate(participacion_grafica.columns):
    fig.add_scatter(
        x=[None],
        y=[None],
        mode="markers",
        name=causa,
        marker=dict(size=10, color=COLORES_TIPOS[i % len(COLORES_TIPOS)]),
        showlegend=True,
    )

fig.update_layout(
    title="Composición anual del monto de siniestros por causa",
    template="reservas",
    barmode="stack",
    showlegend=True,
    barnorm="percent",
    height=700,
    margin=dict(l=65, r=300, t=90, b=65),
    legend=dict(
        orientation="v", x=1.02, xanchor="left", y=1, yanchor="top",
        font=dict(size=11), bgcolor="rgba(242,242,242,0.95)", borderwidth=0,
    ),
    xaxis=dict(title="Año", dtick=1),
    yaxis=dict(title="Participación", ticksuffix="%", range=[0, 100]),
)
fig.show()


#### Ranking acumulado por causa del siniestro

Las barras muestran el monto acumulado completo; las etiquetas incluyen el porcentaje del total acumulado.


In [43]:
from IPython.display import clear_output
clear_output(wait=True)

ranking_grafico = ranking_causas.sort_values("Monto acumulado", ascending=True)
etiquetas = ranking_grafico.apply(
    lambda fila: f"{fila['Monto acumulado']:,.0f}  |  {fila['Participación acumulada (%)']:.1f}%",
    axis=1,
)

fig = go.Figure(go.Bar(
    x=ranking_grafico["Monto acumulado"],
    y=ranking_grafico.index,
    orientation="h",
    marker_color=COLORES["azul"],
    text=etiquetas,
    textposition="outside",
))
fig.update_layout(
    title="Monto acumulado de siniestros por causa",
    template="reservas",
    height=max(520, 22 * len(ranking_grafico)),
    margin=dict(l=220, r=180, t=90, b=65),
    xaxis=dict(title="Monto acumulado (pesos constantes)", tickformat="~s"),
    yaxis=dict(title=""),
)
fig.show()


## 3. Factores de sinestralidad

## 3.1. Sinestralidad total anual

In [44]:
from IPython.display import clear_output
clear_output(wait=True)

from funciones_reservas import (
    factor_siniestralidad_mercado,
    graficar_factor_mercado,
)

# Horizonte maduro: 2015-2022 cuenta con los cuatro desarrollos completos.
anios_completos = list(range(2015, 2023))
factor_mercado = factor_siniestralidad_mercado(
    emision_ajustada,
    sinestralidad_ajustada,
    anios=anios_completos,
)

tabla_mercado = factor_mercado.rename(columns={
    "Año": "Año",
    "prima_emitida": "Prima emitida",
    "monto_siniestro": "Monto del siniestro",
    "factor_siniestralidad": "Factor de siniestralidad",
    "variacion_anual": "Variación anual",
})[[
    "Año", "Prima emitida", "Monto del siniestro",
    "Factor de siniestralidad", "Variación anual",
]]

display(tabla_mercado.style.format({
    "Prima emitida": "{:,.2f}",
    "Monto del siniestro": "{:,.2f}",
    "Factor de siniestralidad": "{:.4f}",
    "Variación anual": "{:+.4f}",
}))

print(
    "Promedio histórico:",
    f"{factor_mercado['factor_siniestralidad'].mean():.4f}",
)
print(
    "Pico:",
    f"{factor_mercado.loc[factor_mercado['factor_siniestralidad'].idxmax(), 'Año']} "
    f"({factor_mercado['factor_siniestralidad'].max():.4f})",
)
print(
    "Valle:",
    f"{factor_mercado.loc[factor_mercado['factor_siniestralidad'].idxmin(), 'Año']} "
    f"({factor_mercado['factor_siniestralidad'].min():.4f})",
)

graficar_factor_mercado(factor_mercado).show()


,Año,Prima emitida,Monto del siniestro,Factor de siniestralidad,Variación anual
0,2015,"2,479,720,780.88","4,285,569,030.11",1.7282,+nan
1,2016,"2,672,242,961.19","1,478,908,507.45",0.5534,-1.1748
2,2017,"3,040,127,080.77","1,021,553,452.08",0.3360,-0.2174
3,2018,"2,549,310,473.56","959,508,132.92",0.3764,+0.0404
4,2019,"3,636,121,176.14","1,390,271,331.35",0.3824,+0.0060
5,2020,"1,677,942,699.91","1,070,848,012.55",0.6382,+0.2558
6,2021,"1,958,679,233.46","1,016,902,419.30",0.5192,-0.1190
7,2022,"1,964,567,043.28","1,322,134,076.59",0.6730,+0.1538


Promedio histórico: 0.6508
Pico: 2015 (1.7282)
Valle: 2017 (0.3360)


## 3.2. Sinestralidad por tipo de embarcación

In [45]:
from IPython.display import clear_output
clear_output(wait=True)

from funciones_reservas import (
    factor_siniestralidad_tipo,
    estadisticas_factor_tipo,
    graficar_factores_tipo,
)

factores_tipo, tipos_activos, primas_cero, primas_tipo = factor_siniestralidad_tipo(
    emision_ajustada,
    sinestralidad_ajustada,
    anios=anios_completos,
)

print("Tipos activos por año")
display(tipos_activos.to_frame().T)

print("Matriz Año × Tipo de embarcación")
display(factores_tipo.style.format("{:.4f}"))

observaciones_prima_cero = []
for año in factores_tipo.index:
    for tipo in factores_tipo.columns:
        if primas_cero.loc[año, tipo]:
            observaciones_prima_cero.append({
                "Año": año,
                "Tipo": tipo,
                    "Factor aplicado": factores_tipo.loc[año, tipo],
            })

observaciones_prima_cero = pd.DataFrame(observaciones_prima_cero)
print("Observaciones con prima cero (factor de pago / 1; excluidas de las visuales)")
if observaciones_prima_cero.empty:
    print("No hay observaciones con prima cero.")
else:
    print(f"Total de observaciones con prima cero: {len(observaciones_prima_cero)}")
    mayores = observaciones_prima_cero.sort_values(
        "Factor aplicado", ascending=False
    ).head(15)
    display(mayores.style.format({"Factor aplicado": "{:.4f}"}))

estadisticas_tipo, cambios_tipo, _ = estadisticas_factor_tipo(
    factores_tipo,
    primas_cero,
)
print("Estadísticos por tipo")
display(estadisticas_tipo.style.format({
    "Mínimo": "{:.4f}",
    "Máximo": "{:.4f}",
    "Media": "{:.4f}",
    "Mediana": "{:.4f}",
    "IQR": "{:.4f}",
    "Atípicos": "{}",
}))

print("Cambios anuales en puntos de factor")
display(cambios_tipo.style.format("{:+.4f}"))

box_factores, heat_factores = graficar_factores_tipo(
    factores_tipo,
    primas_cero,
)
box_factores.show()
heat_factores.show()


Tipos activos por año


año,2015,2016,2017,2018,2019,2020,2021,2022
Tipos activos,24,25,25,26,26,26,26,25


Matriz Año × Tipo de embarcación


tipo_embarcacion,abastecedor,artefactos navales,atunero,barcaza o chalan,barco-escuela,barco-hidrografico,buque frigorifico,buque transportador de autos,buque-tanque,buque-tanque petrolero y productos derivados,buque-tanque quimico,buque-tanque-minerales,camaronero,carguero a granel,carguero general,carguero tipo roll on - roll off,dragas,embarcacion de investigacion,embarcacion menor utilizada para vigilancia,escamero,ferry-transbordador,jet sky o moto acuatica,lancha,militar u oficial,otro tipo de embarcacion,pesca multiple,porta contenedor,remolcador,sardinero o anchovetero,tiburoneros,velero,yate
año,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015,0.2332,-0.0000,0.0459,0.1118,0.0000,0.0000,-0.0000,0.0000,35.7672,0.0007,0.0000,0.0000,0.4266,0.0000,1.4643,0.0000,0.0000,1.5493,-0.0000,0.0000,0.4816,0.0416,0.2244,0.0000,0.0011,0.0091,0.0000,3.4511,0.0167,0.0000,0.3512,0.5469
2016,0.3775,0.0000,0.2674,2.1372,0.0000,0.0000,0.0000,-0.0000,1.0577,0.1227,0.0000,0.0000,1.0469,0.0000,0.1974,0.0000,12.2355,0.4833,0.0000,0.7710,0.7235,0.8075,0.7232,0.0081,10.3791,0.7004,0.0000,0.7495,1.6505,0.0000,0.1127,0.8382
2017,0.4385,0.0000,0.0801,0.0000,0.0000,0.0000,0.0000,0.0000,0.7083,0.3610,0.0000,0.0000,0.8010,0.0000,0.0621,0.0000,0.0000,0.1306,0.0000,0.8084,0.0744,0.0199,0.2926,0.0000,0.1284,0.6083,0.0000,0.9896,0.1605,0.0000,0.0113,0.6997
2018,0.3490,0.0000,0.3712,0.0000,0.0000,0.0000,0.0000,0.0000,3.4706,0.1821,0.0000,0.0000,0.0418,0.0000,0.0824,0.0000,5.2596,0.0159,0.0000,0.1775,0.2592,-120.6483,2.3803,0.0000,0.0619,0.1727,0.0000,0.0613,1.7179,0.0000,0.1419,0.6856
2019,0.0346,335127.4400,0.2641,0.3807,0.0000,0.0000,0.0000,-0.0000,6.8615,0.6556,0.0000,0.0000,0.2414,0.0000,0.9410,0.0000,1.1788,0.1393,0.0000,0.1953,0.0321,0.0925,1.3546,0.0000,6.0567,0.2707,0.0000,0.5513,0.0509,0.0000,0.2094,0.6745
2020,0.5604,5.1443,2.6080,0.0000,0.0000,0.0000,0.0000,0.0000,1.7610,0.0340,0.0000,0.0000,0.4887,0.0000,0.4471,0.0000,0.5687,1.1841,-0.0000,0.0000,0.5023,0.1329,0.1328,0.0000,28.4263,2.6189,0.0000,0.0204,0.0003,0.0000,0.6628,0.6676
2021,0.5710,0.0000,4.0025,2.2376,0.5892,0.0000,0.0000,0.0000,0.1637,2.0388,0.0000,0.0000,0.1144,0.0000,0.0040,0.0000,0.0000,0.5710,67.3701,0.3190,0.2550,1.2499,0.3894,0.0000,0.6674,0.1892,0.0000,0.8478,0.1248,0.0000,1.5239,0.5895
2022,0.5511,0.0000,3.4469,0.0520,-0.0000,0.0000,0.0000,0.0000,0.0654,0.0385,0.0000,0.0000,0.1450,0.0000,0.1457,0.0000,0.0000,0.0462,0.0000,0.0000,0.8704,0.2094,0.0849,0.1059,3.3310,1.1304,0.0000,0.2968,0.4070,0.0000,0.8337,0.6191


Observaciones con prima cero (factor de pago / 1; excluidas de las visuales)
Total de observaciones con prima cero: 45


,Año,Tipo,Factor aplicado
23,2019,artefactos navales,335127.4400
0,2015,barco-hidrografico,0.0000
25,2019,buque-tanque quimico,0.0000
26,2019,carguero tipo roll on - roll off,0.0000
27,2019,tiburoneros,0.0000
28,2020,barco-hidrografico,0.0000
29,2020,buque frigorifico,0.0000
30,2020,buque-tanque quimico,0.0000
31,2020,carguero tipo roll on - roll off,0.0000
32,2020,porta contenedor,0.0000


Estadísticos por tipo


,Mínimo,Máximo,Media,Mediana,IQR,Atípicos,Años atípicos,Atípicos por prima cero
Tipo,,,,,,,,
abastecedor,0.0346,0.5710,0.3894,0.4080,0.2334,0,,Ninguno
artefactos navales,-0.0000,335127.4400,41891.5730,0.0000,1.2861,2,"2019, 2020",2019
atunero,0.0459,4.0025,1.3858,0.3193,2.5996,0,,Ninguno
barcaza o chalan,0.0000,2.2376,0.6149,0.0819,0.8198,2,"2016, 2021",Ninguno
barco-escuela,-0.0000,0.5892,0.0737,0.0000,0.0000,1,2021,Ninguno
barco-hidrografico,0.0000,0.0000,0.0000,0.0000,0.0000,0,,Ninguno
buque frigorifico,-0.0000,0.0000,0.0000,0.0000,0.0000,0,,Ninguno
buque transportador de autos,-0.0000,0.0000,0.0000,0.0000,0.0000,0,,Ninguno
buque-tanque,0.0654,35.7672,6.2319,1.4094,3.7462,1,2015,Ninguno


Cambios anuales en puntos de factor


tipo_embarcacion,abastecedor,artefactos navales,atunero,barcaza o chalan,barco-escuela,barco-hidrografico,buque frigorifico,buque transportador de autos,buque-tanque,buque-tanque petrolero y productos derivados,buque-tanque quimico,buque-tanque-minerales,camaronero,carguero a granel,carguero general,carguero tipo roll on - roll off,dragas,embarcacion de investigacion,embarcacion menor utilizada para vigilancia,escamero,ferry-transbordador,jet sky o moto acuatica,lancha,militar u oficial,otro tipo de embarcacion,pesca multiple,porta contenedor,remolcador,sardinero o anchovetero,tiburoneros,velero,yate
año,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan,+nan
2016,+0.1443,+0.0000,+0.2215,+2.0254,+0.0000,+0.0000,+0.0000,-0.0000,-34.7095,+0.1220,+0.0000,+0.0000,+0.6203,+0.0000,-1.2668,+0.0000,+12.2355,-1.0660,+0.0000,+0.7710,+0.2419,+0.7659,+0.4988,+0.0081,+10.3780,+0.6913,+0.0000,-2.7016,+1.6338,+0.0000,-0.2385,+0.2913
2017,+0.0610,+0.0000,-0.1873,-2.1372,+0.0000,+0.0000,+0.0000,+0.0000,-0.3494,+0.2382,+0.0000,+0.0000,-0.2459,+0.0000,-0.1354,+0.0000,-12.2355,-0.3527,+0.0000,+0.0374,-0.6491,-0.7877,-0.4305,-0.0081,-10.2507,-0.0921,+0.0000,+0.2401,-1.4900,+0.0000,-0.1014,-0.1385
2018,-0.0895,+0.0000,+0.2912,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000,+2.7623,-0.1789,+0.0000,+0.0000,-0.7592,+0.0000,+0.0204,+0.0000,+5.2596,-0.1147,+0.0000,-0.6309,+0.1848,-120.6682,+2.0876,+0.0000,-0.0665,-0.4356,+0.0000,-0.9283,+1.5573,+0.0000,+0.1307,-0.0141
2019,-0.3144,+335127.4400,-0.1071,+0.3807,+0.0000,+0.0000,+0.0000,-0.0000,+3.3909,+0.4735,+0.0000,+0.0000,+0.1996,+0.0000,+0.8586,+0.0000,-4.0809,+0.1234,+0.0000,+0.0178,-0.2272,+120.7408,-1.0257,+0.0000,+5.9948,+0.0980,+0.0000,+0.4900,-1.6670,+0.0000,+0.0674,-0.0110
2020,+0.5258,-335122.2956,+2.3439,-0.3807,+0.0000,+0.0000,+0.0000,+0.0000,-5.1004,-0.6216,+0.0000,+0.0000,+0.2472,+0.0000,-0.4939,+0.0000,-0.6101,+1.0448,-0.0000,-0.1953,+0.4702,+0.0404,-1.2217,+0.0000,+22.3696,+2.3482,+0.0000,-0.5309,-0.0506,+0.0000,+0.4535,-0.0070
2021,+0.0106,-5.1443,+1.3946,+2.2376,+0.5892,+0.0000,+0.0000,+0.0000,-1.5973,+2.0049,+0.0000,+0.0000,-0.3743,+0.0000,-0.4431,+0.0000,-0.5687,-0.6131,+67.3701,+0.3190,-0.2473,+1.1170,+0.2566,+0.0000,-27.7589,-2.4297,+0.0000,+0.8273,+0.1245,+0.0000,+0.8610,-0.0780
2022,-0.0199,+0.0000,-0.5557,-2.1856,-0.5892,+0.0000,+0.0000,+0.0000,-0.0983,-2.0003,+0.0000,+0.0000,+0.0306,+0.0000,+0.1416,+0.0000,+0.0000,-0.5249,-67.3701,-0.3190,+0.6154,-1.0405,-0.3044,+0.1059,+2.6637,+0.9411,+0.0000,-0.5510,+0.2822,+0.0000,-0.6902,+0.0296


## 3.3. Sinestralidad por conglomerados

In [46]:
from IPython.display import clear_output
clear_output(wait=True)

from funciones_reservas import (
    preparar_perfil_conglomerados,
    evaluar_conglomerados,
    asignar_conglomerados,
    graficar_evaluacion_conglomerados,
    graficar_dendrograma,
    graficar_perfiles_conglomerados,
    factor_siniestralidad_conglomerado,
    graficar_factor_conglomerado,
)

tipos_completos = primas_tipo.columns[
    (primas_tipo.loc[anios_completos] > 0).all(axis=0)
].tolist()
print(f"Tipos comparables con prima positiva 2015-2022: {len(tipos_completos)}")
factores_conglomerados = factores_tipo[tipos_completos]
primas_cero_conglomerados = primas_cero[tipos_completos]

perfiles_escalados, perfiles = preparar_perfil_conglomerados(
    factores_conglomerados,
    primas_cero_conglomerados,
)

evaluacion_conglomerados = evaluar_conglomerados(
    perfiles_escalados,
    k_min=2,
    k_max=5,
)
print("Evaluación de k por silueta, Calinski-Harabasz y WCSS")
display(evaluacion_conglomerados.style.format({
    "Silueta": "{:.3f}",
    "Calinski-Harabasz": "{:.2f}",
    "WCSS": "{:.2f}",
}))

graficar_evaluacion_conglomerados(evaluacion_conglomerados).show()
graficar_dendrograma(perfiles_escalados).show()

# Selección reproducible basada en el equilibrio entre silueta y interpretabilidad.
k_seleccionado = 3
asignacion, perfiles_finales = asignar_conglomerados(
    perfiles_escalados,
    perfiles,
    k=k_seleccionado,
)

print("Asignación de tipos a conglomerados")
display(asignacion.sort_values().to_frame())

# Variables disponibles en memoria para cálculos posteriores de reservas
tipos_por_conglomerado = {
    grupo: asignacion[asignacion == grupo].index.tolist()
    for grupo in sorted(asignacion.unique())
}
conglomerados_tipo = asignacion.rename("conglomerado").rename_axis("tipo_embarcacion").to_frame()

print("Asignación disponible en memoria")
print("Variables: asignacion, tipos_por_conglomerado, conglomerados_tipo")
display(conglomerados_tipo.sort_values("conglomerado"))

print("Perfil de los conglomerados")
display(perfiles_finales.style.format({
    "Nivel_promedio": "{:.4f}",
    "Nivel_mediano": "{:.4f}",
    "Variabilidad_promedio": "{:.4f}",
    "Variabilidad_mediana": "{:.4f}",
}))

graficar_perfiles_conglomerados(perfiles, asignacion).show()

# Siniestralidad agregada por conglomerado
factores_conglomerado, perfil_siniestralidad = factor_siniestralidad_conglomerado(
    factores_conglomerados,
    primas_tipo.reindex(index=factores_conglomerados.index, columns=tipos_completos),
    asignacion,
)

print("Factor anual agregado por conglomerado")
display(factores_conglomerado.style.format("{:.4f}"))

print("Resumen histórico de siniestralidad por conglomerado")
display(perfil_siniestralidad.style.format({
    "Prima agregada": "{:,.2f}",
    "Siniestro agregado": "{:,.2f}",
    "Factor agregado": "{:.4f}",
    "Factor anual medio": "{:.4f}",
    "Factor anual mediano": "{:.4f}",
    "Volatilidad (DE)": "{:.4f}",
}))

graficar_factor_conglomerado(factores_conglomerado).show()


Tipos comparables con prima positiva 2015-2022: 19
Evaluación de k por silueta, Calinski-Harabasz y WCSS


,k,Silueta,Calinski-Harabasz,WCSS
0,2,0.489,17.29,18.84
1,3,0.368,18.74,11.37
2,4,0.372,20.76,7.37
3,5,0.356,20.67,5.50


Asignación de tipos a conglomerados


,Conglomerado
tipo_embarcacion,
abastecedor,G1
ferry-transbordador,G1
escamero,G1
carguero a granel,G1
camaronero,G1
yate,G1
carguero general,G2
velero,G2
barcaza o chalan,G2


Asignación disponible en memoria
Variables: asignacion, tipos_por_conglomerado, conglomerados_tipo


,conglomerado
tipo_embarcacion,
abastecedor,G1
ferry-transbordador,G1
escamero,G1
carguero a granel,G1
camaronero,G1
yate,G1
carguero general,G2
velero,G2
barcaza o chalan,G2


... (2 outputs left out)

Factor anual agregado por conglomerado


,G1,G2,G3
año,,,
2015,0.2706,0.1363,10.2565
2016,0.4387,0.3953,2.3025
2017,0.4714,0.3121,0.1923
2018,0.4134,0.3076,0.2722
2019,0.0717,0.8449,2.3966
2020,0.5760,0.6208,2.8888
2021,0.5515,0.3407,0.7562
2022,0.5629,0.4560,1.9432


Resumen histórico de siniestralidad por conglomerado


,Prima agregada,Siniestro agregado,Factor agregado,Factor anual medio,Factor anual mediano,Volatilidad (DE)
Conglomerado,,,,,,
G1,"8,936,018,590.26","2,992,189,778.64",0.3348,0.4195,0.4551,0.1727
G2,"7,513,973,214.11","2,935,828,589.59",0.3907,0.4267,0.3680,0.2183
G3,"2,788,471,190.77","6,182,287,365.62",2.2171,2.6260,2.1228,3.2486


# 4. Reservas

## 4.2. Reserva de Riesgos en Curso

## 4.3. Reserva de Obligaciones Pendientes de Cumplir por Siniestros Ocurridos y No Reportados

# 5. Conclusiones